# Сборка единого датасета для обучения модели.

## Plan:

- Загрузить данные.
- Отобрать бассейны.
- Объединить в один датасет с фичами для обучения моделей.
- Обучить модели.

In [1]:
import os
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import seaborn as sns
from tqdm.notebook import tqdm

from utils.datasets import (
    ARTIFACTS_FOLDER,
    ROOT_FOLDER,
    STATIC_FEATURES,
    HydroFiles,
    HydroStaticFeaturesFiles,
    MeteoSeriesFeaturesFiles,
)
from utils.types import TimeRange
from pathlib import Path

os.chdir(ROOT_FOLDER)

In [2]:
ARTIFACTS_FOLDER

PosixPath('/home/khuzin/Projects/2025-m1p-private/data/artifacts')

## Global constants

In [3]:
# Horizons for forecasting
HISTORY_HORIZON = TimeRange.YEAR
FORECAST_HORIZON = TimeRange.WEEK

# Columns setups
COL_TARGET = "q_mm_day"
COL_GAUGE_ID = "gauge_id"

# Lags
FEATURES_LAGS = None  # Filled below.

## Datasets correction

In [4]:
msff = MeteoSeriesFeaturesFiles()
hf = HydroFiles()

In [5]:
msff_ids = msff.get_ids()
hf_ids = hf.get_ids()
union_ids = list(set(msff_ids) & set(hf_ids))

len(msff_ids), len(hf_ids), len(union_ids)

(2201, 2203, 2201)

In [6]:
MSFF_DROP_SET = set(
    [
        "gauge_id",
        "q_cms",
        "q_cms_s",
        "q_mm_day",
        "lvl_sm",
        "lvl_mbs",
        "day_of_year",
        "lvl_mbs",
    ]
)

HF_DROP_SET = set(["q_cms", "q_cms_right", "lvl_mbs"])

In [7]:
def prepare_msff_dataset(ds: pl.LazyFrame):
    DROP_COLS = MSFF_DROP_SET & set(ds.collect_schema().names())
    return ds.drop(DROP_COLS)


def prepare_hf_dataset(ds: pl.LazyFrame):
    DROP_COLS = HF_DROP_SET & set(ds.collect_schema().names())
    return ds.drop(DROP_COLS)

In [8]:
SAVE_PATH = ARTIFACTS_FOLDER / "merged_datasets"
SAVE_PATH.mkdir(parents=True, exist_ok=True)


merged_datasets = dict()
count = 0
training_subset = list()

for file_id in tqdm(union_ids, desc="Merging files"):
    pre_msff = prepare_msff_dataset(msff[file_id])
    pre_hf = prepare_hf_dataset(hf[file_id])
    merged = pre_msff.join(pre_hf, on="date")
    merged = merged.with_columns(pl.lit(file_id).alias("gauge_id"))
    merged_datasets[file_id] = merged

for file_id, dataset in tqdm(merged_datasets.items()):
    dataset.collect().write_parquet(SAVE_PATH / f"{file_id}.parquet")

for file_id, dataset in tqdm(merged_datasets.items()):
    if (
        dataset.filter(pl.any_horizontal(pl.all().is_null()))
        .collect()
        .height
        != 0
    ):
        count += 1
        training_subset.append(file_id)

pl.DataFrame({"file_id": training_subset}).write_csv(SAVE_PATH / "train file_ids.csv")
print(count)

Merging files:   0%|          | 0/2201 [00:00<?, ?it/s]

  0%|          | 0/2201 [00:00<?, ?it/s]

  0%|          | 0/2201 [00:00<?, ?it/s]

1129


In [9]:
from joblib import Parallel, delayed

SAVE_PATH = ARTIFACTS_FOLDER / "merged_datasets"
N_JOBS = -1


SAVE_PATH.mkdir(parents=True, exist_ok=True)


def process_and_check(file_id):
    left = prepare_msff_dataset(msff[file_id])
    right = prepare_hf_dataset(hf[file_id])
    merged = left.join(right, on="date")
    merged = merged.with_columns(pl.lit(file_id).alias("gauge_id"))

    output_path = SAVE_PATH / f"{file_id}.parquet"
    merged.collect().write_parquet(output_path)

    has_nulls = (
        merged.filter(pl.any_horizontal(pl.all().is_null())).collect().height != 0
    )

    return file_id, has_nulls


results = Parallel(n_jobs=N_JOBS)(
    delayed(process_and_check)(fid) for fid in tqdm(union_ids, desc="Processing files")
)

training_subset = [fid for fid, has_null in results if (has_null != True)]
count = len(training_subset)

pl.DataFrame({"file_id": training_subset}).write_csv(SAVE_PATH / "train_file_ids.csv")
print(f"Files with nulls: {count}")

Processing files:   0%|          | 0/2201 [00:00<?, ?it/s]

Files with nulls: 1072


----------------------------------------------

----------------------------------------------

----------------------------------------------

----------------------------------------------

## Create a dataset for CatBoost

In [3]:
SAVE_PATH = ARTIFACTS_FOLDER / "merged_datasets"
merged_datasets = dict()

for filename in tqdm(os.listdir(SAVE_PATH)):
    gauge_id = int(filename.split(".")[0])
    merged_datasets[gauge_id] = pl.read_parquet(SAVE_PATH / filename)

  0%|          | 0/2201 [00:00<?, ?it/s]

In [4]:
merged_datasets[gauge_id].head(2)

date,prcp,t_max,t_mean,t_min,q_mm_day,lvl_sm
date,f64,f64,f64,f64,f64,f64
2008-01-01,0.061883,-15.777812,-18.836736,-21.852113,0.137761,120.0
2008-01-02,0.4825,-9.904813,-13.245992,-18.357174,0.141049,122.0


### Add lag features

In [5]:
TARGETS = ["qq_mm_day"]

LAG_FEATUES = {
    col: range(1, TimeRange.YEAR + 1)
    for col in next(iter(merged_datasets.values())).columns
    if col != "date"
}
print([name for name in LAG_FEATUES])

['prcp', 't_mean', 't_min', 't_max', 'q_mm_day', 'lvl_sm']


In [6]:
def add_lag_features(
    dataset: pl.DataFrame,
    lag_features: Dict[str, int] = LAG_FEATUES,
) -> pl.LazyFrame:
    """
    Adds lagged columns for specified features in a LazyFrame.
    """
    exprs = []
    ds_columns = list(dataset.schema)
    for col, lag_range in lag_features.items():
        if col in ds_columns:
            for lag in lag_range:
                exprs.append(pl.col(col).shift(lag).alias(f"{col}_lag_{lag}"))
    return dataset.with_columns(exprs)


def add_static_features(
    dataset: pl.LazyFrame,
    static_features: pl.DataFrame,
) -> pl.LazyFrame:
    static_dict = static_features.to_dict()
    for col, value in static_dict.items():
        dataset = dataset.with_columns(pl.lit(value).first().alias(col))
    return dataset

#### Корреляционный анализ.

In [60]:
flag = 0
eps = 1e-9
for value in merged_datasets.values():
    if td[["lvl_sm", "lvl_mbs"]].drop_nulls().height != 0:
        if (
            td[["lvl_sm", "lvl_mbs"]]
            .select(pl.corr(pl.col("lvl_sm"), pl.col("lvl_mbs")))
            .item()
            < 1 - eps
        ):
            flag += 1

flag

0

Видим совершенную корреляцию. (Код выше отредактирован)

### Создание итогового датасета

In [17]:
hsff = HydroStaticFeaturesFiles()
hsff_ids = set(hsff.get_ids())
hsff = hsff.dataframe.select(STATIC_FEATURES + ["gauge_id"]).collect()

merged_ids = set(merged_datasets.keys())
union_ids = hsff_ids & merged_ids
print(len(hsff_ids), len(merged_ids), len(union_ids))

2203 2201 2201


In [21]:
def create_final_dataset(file_id: int) -> pl.DataFrame:
    res = add_static_features(
        merged_datasets[file_id], hsff.filter(pl.col("gauge_id") == file_id)
    )
    res = add_lag_features(res)
    return res

In [25]:
SAVE_FINAL = ARTIFACTS_FOLDER / "catboots_ds"
SAVE_FINAL.mkdir(parents=True, exist_ok=True)

count = 0
for file_id in tqdm(union_ids):
    if (
        merged_datasets[file_id]
        .filter(pl.any_horizontal(pl.all().is_null()))
        .height
        != 0
    ):
        count += 1
        merged_datasets[file_id] = create_final_dataset(file_id)

print(count)

  0%|          | 0/2201 [00:00<?, ?it/s]

1129
